# Instant4D Native Colab Run, Python 3.10 / Torch 2.2

This notebook is for the real comparison run. It avoids the previous Colab fallback path by creating an isolated Python 3.10 environment with native `xformers==0.0.24` and native `torch-scatter` for `torch==2.2.0+cu121`.

## Runtime

Use a GPU runtime. A100 is the safest target. H100 should also work with the default `TORCH_CUDA_ARCH_LIST=8.0;9.0`, but A100 is closer to the environment we have been debugging.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi
!python --version

## Fresh Clone

This uses a separate folder from the earlier fallback test so generated fallback files cannot shadow native wheels.

In [ ]:
REPO_URL = 'https://github.com/chiou1203/Instant4D.git'
BRANCH = 'codex-colab-a100-workflow'
REPO_DIR = '/content/Instant4D-native'

%cd /content
!rm -rf {REPO_DIR}
!git clone --recursive --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse --short HEAD

## Native Environment Setup

This creates `/content/micromamba` + env `instant4d310`, installs the native pinned stack, downloads checkpoints, applies only the Mega-SAM Torch API source patch, verifies native imports, then builds CUDA extensions.

In [ ]:
!REPO_ROOT={REPO_DIR} ENV_NAME=instant4d310 MAMBA_ROOT_PREFIX=/content/micromamba bash colab/setup_native_colab.sh

In [ ]:
!MAMBA_ROOT_PREFIX=/content/micromamba /content/micromamba-bin/micromamba run -n instant4d310 python colab/verify_native_deps.py

## Run Full Pipeline

Start with the bundled `panda` sequence. For a Drive dataset, set `DATA_DIR='/content/drive/MyDrive/Instant4D/datasets'` and place frames under `DATA_DIR/SCENE_NAME/`.

In [ ]:
SCENE_NAME = 'panda'
DATA_DIR = f'{REPO_DIR}/example'
CONFIG_PATH = f'{REPO_DIR}/configs/sora/panda.yaml'
MODEL_DIR = f'{REPO_DIR}/output/native/{SCENE_NAME}'
LOG_DIR = '/content/drive/MyDrive/Instant4D/logs'
LOG_PATH = f'{LOG_DIR}/native_{SCENE_NAME}.log'
!mkdir -p {LOG_DIR}

In [ ]:
!MAMBA_ROOT_PREFIX=/content/micromamba REPO_ROOT={REPO_DIR} SCENE_NAME={SCENE_NAME} DATA_DIR={DATA_DIR} CONFIG_PATH={CONFIG_PATH} MODEL_DIR={MODEL_DIR} UNIDEPTH_MODEL_PATH={REPO_DIR}/checkpoints/hf/unidepth-v2-vitl14 /content/micromamba-bin/micromamba run -n instant4d310 bash colab/run_native_colab.sh 2>&1 | tee {LOG_PATH}

## Post-Run Sanity Check

This cell should show `Training complete` and should not show `Traceback`, `ModuleNotFoundError`, or persistent `Lssim=nan`.

In [ ]:
!grep -E "Training complete|Loss=nan|Lssim=nan|Traceback|ModuleNotFoundError|RuntimeError|Could not open|ERROR" {LOG_PATH} || true
!find {MODEL_DIR} -maxdepth 3 -type f | sort | tail -50

## Persist Outputs

Copy the model output folder to Drive after a successful run.

In [ ]:
!mkdir -p /content/drive/MyDrive/Instant4D/outputs/native
!cp -r {MODEL_DIR} /content/drive/MyDrive/Instant4D/outputs/native/